# Teacher Summary Inference với Qwen3-8B

Notebook này chỉ chạy inference để tạo `summary` và `evidence` cho 20 record trong `summary_review.json`. Notebook không fine-tune model, không phê duyệt target và không thay đổi dataset v1.

Kết quả hợp lệ được ghi vào `teacher_outputs.json`. Mẫu lỗi được ghi riêng vào `teacher_errors.json`.

## 1. Chọn GPU trên Google Colab

Chọn **Runtime → Change runtime type → Hardware accelerator → GPU**. T4 có thể chạy Qwen3-8B ở 4-bit; L4 hoặc A100 sẽ nhanh hơn. Sau đó chạy cell kiểm tra GPU bên dưới.

In [ ]:
!pip -q install -U "transformers>=4.51.0" accelerate bitsandbytes tqdm

In [ ]:
import json
from pathlib import Path

import torch
from google.colab import drive, files
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen3-8B'
USE_4BIT = True
MAX_INPUT_TOKENS = 30_000
MAX_NEW_TOKENS = 768

if not torch.cuda.is_available():
    raise RuntimeError('Chưa có GPU. Hãy chọn GPU trong Runtime settings.')

GPU_NAME = torch.cuda.get_device_name(0)
COMPUTE_DTYPE = (
    torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
print('GPU:', GPU_NAME)
print('Compute dtype:', COMPUTE_DTYPE)

## 2. Cung cấp `summary_review.json`

- Cách đơn giản: giữ `USE_GOOGLE_DRIVE = False`, chạy cell và upload file.
- Cách an toàn hơn khi Colab có thể ngắt: upload file vào Google Drive, đặt `USE_GOOGLE_DRIVE = True` và sửa `DRIVE_FOLDER`. Kết quả checkpoint sẽ được lưu trực tiếp vào Drive.

In [ ]:
USE_GOOGLE_DRIVE = False
DRIVE_FOLDER = '/content/drive/MyDrive/KLTN/data_v2'

if USE_GOOGLE_DRIVE:
    drive.mount('/content/drive')
    output_dir = Path(DRIVE_FOLDER)
    queue_path = output_dir / 'summary_review.json'
else:
    uploaded_files = files.upload()
    if not uploaded_files:
        raise FileNotFoundError('Chưa upload summary_review.json')
    uploaded_name = next(iter(uploaded_files))
    queue_path = Path('/content') / uploaded_name
    output_dir = Path('/content/teacher_summary_outputs')

output_dir.mkdir(parents=True, exist_ok=True)
outputs_path = output_dir / 'teacher_outputs.json'
errors_path = output_dir / 'teacher_errors.json'
print('Input:', queue_path)
print('Output folder:', output_dir)

In [ ]:
review_queue = json.loads(queue_path.read_text(encoding='utf-8'))
if not isinstance(review_queue, list) or not review_queue:
    raise ValueError('summary_review.json phải là list không rỗng')

sample_ids = []
for index, record in enumerate(review_queue):
    if not isinstance(record, dict):
        raise ValueError(f'Record {index} phải là object')
    sample_id = record.get('sample_id')
    teacher_prompt = record.get('teacher_prompt')
    if not isinstance(sample_id, str) or not sample_id:
        raise ValueError(f'Record {index} thiếu sample_id')
    if not isinstance(teacher_prompt, str) or not teacher_prompt.strip():
        raise ValueError(f'{sample_id} thiếu teacher_prompt')
    sample_ids.append(sample_id)

if len(sample_ids) != len(set(sample_ids)):
    raise ValueError('summary_review.json có sample_id bị trùng')

print('Số record:', len(review_queue))
print('Sample đầu tiên:', sample_ids[0])

## 3. Load Qwen3-8B

Mặc định notebook dùng 4-bit để giảm VRAM. Đây chỉ là lượng tử hóa phục vụ inference, không phải QLoRA training.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

quantization_config = None
if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
        bnb_4bit_use_double_quant=True,
    )

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=COMPUTE_DTYPE,
    device_map='auto',
    quantization_config=quantization_config,
)
model.eval()
print('Model loaded:', MODEL_ID)
print('Memory footprint (GB):', round(model.get_memory_footprint() / 1024**3, 2))

## 4. Hàm inference, parse JSON và checkpoint

`enable_thinking=False` tắt thinking mode của Qwen3. `do_sample=False` giúp kết quả dễ tái lập. Output sai JSON hoặc evidence không trích từ Master CV/user_info sẽ được ghi vào file lỗi, không được thay bằng summary cũ.

In [ ]:
def read_existing_results(path):
    if not path.exists():
        return {}
    rows = json.loads(path.read_text(encoding='utf-8'))
    return {row['sample_id']: row for row in rows}


def write_checkpoint(path, rows):
    # Ghi file tạm trước để giảm nguy cơ hỏng file khi Colab ngắt.
    temporary_path = path.with_suffix(path.suffix + '.tmp')
    content = json.dumps(list(rows.values()), ensure_ascii=False, indent=2) + '\n'
    temporary_path.write_text(content, encoding='utf-8')
    temporary_path.replace(path)


def parse_teacher_input(teacher_prompt):
    marker = '\n\nINPUT:\n'
    if marker not in teacher_prompt:
        raise ValueError('teacher_prompt không có phần INPUT')
    input_text = teacher_prompt.split(marker, 1)[1]
    teacher_input = json.loads(input_text)
    if not isinstance(teacher_input, dict):
        raise ValueError('INPUT trong teacher_prompt phải là object')
    return teacher_input


def collect_strings(value):
    if isinstance(value, str):
        yield value
    elif isinstance(value, dict):
        for item in value.values():
            yield from collect_strings(item)
    elif isinstance(value, list):
        for item in value:
            yield from collect_strings(item)


def strip_code_fence(text):
    text = text.strip()
    if not text.startswith('```'):
        return text
    lines = text.splitlines()
    if len(lines) < 3 or lines[-1].strip() != '```':
        raise ValueError('JSON code fence không đóng đúng cách')
    return '\n'.join(lines[1:-1]).strip()


def parse_model_output(raw_output, teacher_prompt):
    parsed = json.loads(strip_code_fence(raw_output))
    if not isinstance(parsed, dict):
        raise ValueError('Teacher output phải là JSON object')

    summary = parsed.get('summary')
    evidence = parsed.get('evidence')
    if not isinstance(summary, str) or not summary.strip():
        raise ValueError('summary phải là chuỗi không rỗng')
    if not isinstance(evidence, list) or not evidence:
        raise ValueError('evidence phải là list không rỗng')

    teacher_input = parse_teacher_input(teacher_prompt)
    source_texts = list(collect_strings([
        teacher_input.get('original_cv'),
        teacher_input.get('user_info'),
    ]))
    clean_evidence = []
    for index, item in enumerate(evidence):
        if not isinstance(item, dict):
            raise ValueError(f'evidence[{index}] phải là object')
        claim = item.get('claim')
        source = item.get('source')
        if not isinstance(claim, str) or not claim.strip():
            raise ValueError(f'evidence[{index}].claim không hợp lệ')
        if not isinstance(source, str) or not source.strip():
            raise ValueError(f'evidence[{index}].source không hợp lệ')
        if not any(source.strip() in text for text in source_texts):
            raise ValueError(
                f'evidence[{index}].source không xuất hiện trong Master CV/user_info'
            )
        clean_evidence.append({
            'claim': claim.strip(),
            'source': source.strip(),
        })

    return {'summary': summary.strip(), 'evidence': clean_evidence}


def generate_one(teacher_prompt):
    messages = [{'role': 'user', 'content': teacher_prompt}]
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    model_inputs = tokenizer(formatted_prompt, return_tensors='pt').to(model.device)
    input_length = model_inputs['input_ids'].shape[-1]
    if input_length > MAX_INPUT_TOKENS:
        raise ValueError(
            f'Prompt có {input_length} tokens, vượt giới hạn {MAX_INPUT_TOKENS}'
        )

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = generated_ids[0, input_length:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def run_inference(records, limit):
    selected_records = records[:limit]
    outputs_by_id = read_existing_results(outputs_path)
    errors_by_id = read_existing_results(errors_path)

    for record in tqdm(selected_records, desc=f'Qwen3-8B: {limit} mẫu'):
        sample_id = record['sample_id']
        if sample_id in outputs_by_id:
            continue

        raw_output = None
        try:
            raw_output = generate_one(record['teacher_prompt'])
            parsed_output = parse_model_output(raw_output, record['teacher_prompt'])
            outputs_by_id[sample_id] = {
                'sample_id': sample_id,
                'summary': parsed_output['summary'],
                'evidence': parsed_output['evidence'],
            }
            errors_by_id.pop(sample_id, None)
        except (ValueError, json.JSONDecodeError, RuntimeError) as error:
            errors_by_id[sample_id] = {
                'sample_id': sample_id,
                'error': str(error),
                'raw_output': raw_output,
            }

        # Checkpoint sau từng mẫu để giảm mất dữ liệu khi runtime ngắt.
        write_checkpoint(outputs_path, outputs_by_id)
        write_checkpoint(errors_path, errors_by_id)

    print('Teacher outputs:', len(outputs_by_id), outputs_path)
    print('Teacher errors:', len(errors_by_id), errors_path)
    return list(outputs_by_id.values()), list(errors_by_id.values())

## 5. Chạy thử 3 mẫu

Đọc ba kết quả và file lỗi trước khi chạy đủ 20 mẫu. Không chỉnh `summary_review.json` trong notebook này.

In [ ]:
test_outputs, test_errors = run_inference(review_queue, limit=3)
print(json.dumps(test_outputs, ensure_ascii=False, indent=2))
if test_errors:
    print('Lỗi cần kiểm tra:')
    print(json.dumps(test_errors, ensure_ascii=False, indent=2))

## 6. Chạy đủ 20 mẫu

Cell này tiếp tục từ checkpoint và tự bỏ qua ba mẫu đã thành công. Mẫu từng lỗi sẽ được thử lại.

In [ ]:
teacher_outputs, teacher_errors = run_inference(review_queue, limit=20)

## 7. Tải kết quả về máy

`teacher_outputs.json` tương thích với `--import-teacher`. `teacher_errors.json` chỉ dùng để xem lỗi và không được import. Nếu dùng Google Drive, hai file cũng đã nằm trong `DRIVE_FOLDER`.

In [ ]:
if outputs_path.exists():
    files.download(str(outputs_path))
if errors_path.exists():
    files.download(str(errors_path))